# PJMW Hourly Electricity Demand Forecasting
**Model Building & Development** (one-by-one, correct order)

| Step | Section |
|---|---|
| 0 | Install & Imports |
| 1 | Load data |
| 2 | Data understanding (EDA checks) |
| 3 | Feature validation (leakage check) |
| 4 | Train / Validation / Test split (time-based) |
| 5 | Evaluation function |
| 6 | Baseline model (Naive) |
| 7 | Machine-learning models: Linear Regression, Random Forest, XGBoost, LightGBM |
| 8 | Time-series cross-validation |
| 9 | Hyperparameter tuning (XGBoost) |
| 10 | Ensemble (XGBoost tuned + LightGBM) |
| 11 | Classical time-series models: ARIMA & SARIMA |
| 12 | Final model comparison |
| 13 | Best model analysis (plots, residuals, error by hour/month/day, feature importance) |
| 14 | Final model training & saving |
| 15 | Prediction on new data (inference) |

## Step 0: Install & Imports

In [ ]:
!pip install xgboost lightgbm statsmodels openpyxl

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, cross_val_score, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor
import lightgbm as lgb

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
RANDOM_STATE = 42

## Step 1: Load data

In [ ]:
df = pd.read_excel("PJMW_features.xlsx")
df.head()

## Step 2: Data understanding
Shape, data types, nulls, duplicates, statistics.

In [ ]:
print("Shape     :", df.shape)
print("Nulls     :", df.isnull().sum().sum())
print("Duplicates:", df.duplicated().sum())

In [ ]:
df.info()

In [ ]:
df.describe().T

## Step 3: Feature validation (leakage check)
`rolling_24` must **not** include the current value (that would leak the target). It should use previous values only (`shift(1)`).
Here we check whether the `rolling_24` column matches the `shift(1)` version.

In [ ]:
y_all = df["PJMW_MW"]

no_shift   = y_all.rolling(24).mean()             # leaky version (includes the current value)
with_shift = y_all.shift(1).rolling(24).mean()    # correct version (previous values only)

# skip the first 168 rows (the lag_168 warm-up rows that were dropped)
a = df["rolling_24"].iloc[168:]
print("Matches shift(1) version :", np.allclose(a, with_shift.iloc[168:], rtol=1e-3))
print("Matches no-shift version :", np.allclose(a, no_shift.iloc[168:], rtol=1e-3))

## Step 4: Train / Validation / Test split (time-based)
Because this is a time series, we **must not split randomly** - the split is chronological.
- **Train** = first 80% (its last 10% is the **validation** set, used for early stopping)
- **Test**  = last 20% (used only for the final evaluation, never for training)

In [ ]:
target = "PJMW_MW"
X = df.drop(columns=[target])
y = df[target]
print(X.shape, y.shape)

# Train / Test
split = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

# Validation set = last 10% of train (used for early stopping)
val_split = int(len(X_train) * 0.9)
X_tr, X_val = X_train.iloc[:val_split], X_train.iloc[val_split:]
y_tr, y_val = y_train.iloc[:val_split], y_train.iloc[val_split:]

print("Train years:", X_train["year"].min(), "-", X_train["year"].max())
print("Test years :", X_test["year"].min(), "-", X_test["year"].max())
print("Sizes -> train(sub):", len(X_tr), " val:", len(X_val), " test:", len(X_test))

## Step 5: Evaluation function

In [ ]:
def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    print(f"{name:20s} MAE={mae:8.2f}  RMSE={rmse:8.2f}  MAPE={mape:5.2f}%  R2={r2:.4f}")
    return {"model": name, "MAE": mae, "RMSE": rmse, "MAPE": mape, "R2": r2}

results = []       # metrics for the comparison table
predictions = {}   # predictions of every model (used for plots)

def log_result(name, pred):
    pred = np.asarray(pred, dtype=float)
    predictions[name] = pred
    results.append(evaluate(name, y_test, pred))

## Step 6: Baseline model (Naive)
Yesterday's demand at the same hour = today's demand. Every other model should beat this.

In [ ]:
log_result("Naive (lag_24)", X_test["lag_24"])

## Step 7: Machine-learning models

### 7.1 Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
log_result("Linear Regression", lr.predict(X_test))

### 7.2 Random Forest

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=RANDOM_STATE
)
rf.fit(X_train, y_train)
log_result("Random Forest", rf.predict(X_test))

### 7.3 XGBoost
Early stopping uses the **validation set** (not the test set).

In [ ]:
xgb = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    early_stopping_rounds=50,
    n_jobs=-1,
    random_state=RANDOM_STATE
)
xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=100)
print("Best iteration:", xgb.best_iteration)
log_result("XGBoost", xgb.predict(X_test))

### 7.4 LightGBM
Early stopping uses the **validation set**.

In [ ]:
lgbm = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)
lgbm.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(200)]
)
log_result("LightGBM", lgbm.predict(X_test))

## Step 8: Time-series cross-validation
`TimeSeriesSplit` always trains on the past and validates on the future. It is run on the **training data only** (the test set is never touched).

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

cv_model = XGBRegressor(
    n_estimators=300, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=RANDOM_STATE
)

scores = cross_val_score(cv_model, X_train, y_train, cv=tscv,
                         scoring="neg_root_mean_squared_error")
print("RMSE per fold:", -scores)
print("Mean RMSE    :", -scores.mean())
print("Std          :", scores.std())

## Step 9: Hyperparameter tuning (XGBoost)
RandomizedSearchCV + TimeSeriesSplit, run on the **training data only**.

In [ ]:
param_dist = {
    "n_estimators":     [200, 400, 600, 800],
    "learning_rate":    [0.01, 0.03, 0.05, 0.1],
    "max_depth":        [4, 6, 8, 10],
    "min_child_weight": [1, 3, 5, 10],
    "subsample":        [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_lambda":       [0.5, 1, 5, 10],
}

search = RandomizedSearchCV(
    XGBRegressor(n_jobs=-1, random_state=RANDOM_STATE),
    param_distributions=param_dist,
    n_iter=20,
    cv=TimeSeriesSplit(n_splits=3),
    scoring="neg_root_mean_squared_error",
    verbose=2,
    random_state=RANDOM_STATE,
    n_jobs=1
)
search.fit(X_train, y_train)

print("Best params:", search.best_params_)

In [ ]:
best_xgb = search.best_estimator_
log_result("XGBoost (tuned)", best_xgb.predict(X_test))

## Step 10: Ensemble (XGBoost tuned + LightGBM)

In [ ]:
pred_ens = (predictions["XGBoost (tuned)"] + predictions["LightGBM"]) / 2
log_result("Ensemble (XGB+LGB)", pred_ens)

## Step 11: Classical time-series models - ARIMA & SARIMA
ARIMA/SARIMA are **univariate** models: they use only the past values of `PJMW_MW` (not features such as `lag_*` or `hour_sin`).
- Fitting SARIMA on the full training data (100k+ hours) is very slow, so we fit on the **last 90 days of the training data**.
- Evaluation is **one-step-ahead**: the fitted parameters stay fixed and every hour is predicted from the previous actual values (comparable to the ML models that use `lag_1`).

In [ ]:
TRAIN_WINDOW = 24 * 90                       # last 90 days of train data
y_tr_tail = y_train.values[-TRAIN_WINDOW:].astype(float)
y_te      = y_test.values.astype(float)

def one_step_ahead(fitted_res, y_new, chunk=1000):
    """One-step-ahead predictions on the test data with fixed parameters (processed in chunks to save memory)."""
    res, out = fitted_res, []
    for i in range(0, len(y_new), chunk):
        res = res.extend(y_new[i:i+chunk])   # the state continues from the end of the previous chunk
        out.append(np.asarray(res.fittedvalues))
    return np.concatenate(out)

### 11.1 ARIMA(2,1,2)

In [ ]:
t0 = time.time()
arima_fit = ARIMA(y_tr_tail, order=(2, 1, 2)).fit()
print(arima_fit.summary().tables[1])

log_result("ARIMA(2,1,2)", one_step_ahead(arima_fit, y_te))
print(f"Time: {time.time()-t0:.1f}s")

### 11.2 SARIMA(1,0,1)x(1,1,1,24)
Seasonal period 24 = daily seasonality.

In [ ]:
t0 = time.time()
sarima_fit = SARIMAX(
    y_tr_tail,
    order=(1, 0, 1),
    seasonal_order=(1, 1, 1, 24),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False, maxiter=50)
print(sarima_fit.summary().tables[1])

log_result("SARIMA(24)", one_step_ahead(sarima_fit, y_te))
print(f"Time: {time.time()-t0:.1f}s")

## Step 12: Final model comparison

In [ ]:
final_table = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
final_table

In [ ]:
plt.figure(figsize=(10, 4))
plt.barh(final_table["model"][::-1], final_table["RMSE"][::-1])
plt.xlabel("RMSE (MW) - lower is better")
plt.title("Model comparison")
plt.tight_layout()
plt.show()

## Step 13: Best model analysis
The model with the **lowest RMSE** in the final table is selected automatically as the best model.

In [ ]:
best_name = final_table.loc[0, "model"]
best_pred = predictions[best_name]
print("Best model:", best_name)

### 13.1 Actual vs Predicted

In [ ]:
n = 24 * 14
plt.figure(figsize=(14, 5))
plt.plot(y_test.values[:n], label="Actual")
plt.plot(best_pred[:n], label=f"Predicted ({best_name})")
plt.title("Actual vs Predicted (first 14 days of test)")
plt.xlabel("Hours")
plt.ylabel("MW")
plt.legend()
plt.show()

### 13.2 ARIMA vs SARIMA vs Actual

In [ ]:
n = 24 * 7
plt.figure(figsize=(14, 5))
plt.plot(y_te[:n], label="Actual", linewidth=2)
plt.plot(predictions["ARIMA(2,1,2)"][:n], label="ARIMA(2,1,2)", alpha=0.8)
plt.plot(predictions["SARIMA(24)"][:n],   label="SARIMA(24)",   alpha=0.8)
plt.title("ARIMA vs SARIMA (first 7 days of test, one-step-ahead)")
plt.xlabel("Hours"); plt.ylabel("MW"); plt.legend()
plt.show()

### 13.3 Residual (error) analysis

In [ ]:
residuals = y_test.values - best_pred

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.hist(residuals, bins=50)
plt.title("Error distribution")

plt.subplot(1, 2, 2)
plt.scatter(best_pred, residuals, s=2, alpha=0.3)
plt.axhline(0, color="red")
plt.title("Predicted vs Error")
plt.show()

### 13.4 Error by hour / month / day of week

In [ ]:
err = pd.DataFrame({
    "hour":  X_test["hour"].values,
    "month": X_test["month"].values,
    "dow":   X_test["day_of_week"].values,
    "abs_err": np.abs(y_test.values - best_pred)
})

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
err.groupby("hour")["abs_err"].mean().plot(ax=ax[0], title="MAE by hour")
err.groupby("month")["abs_err"].mean().plot(kind="bar", ax=ax[1], title="MAE by month")
err.groupby("dow")["abs_err"].mean().plot(kind="bar", ax=ax[2], title="MAE by day of week")
plt.tight_layout()
plt.show()

### 13.5 Feature importance (tuned XGBoost)

In [ ]:
imp = pd.Series(best_xgb.feature_importances_, index=X.columns).sort_values()
imp.plot(kind="barh", figsize=(8, 6), title="Feature Importance (XGBoost tuned)")
plt.show()

## Step 14: Final model training & saving
The tuned XGBoost is retrained on the **full data (train + test)** and saved.

In [ ]:
final_model = XGBRegressor(**search.best_params_, n_jobs=-1, random_state=RANDOM_STATE)
final_model.fit(X, y)

joblib.dump(final_model, "pjmw_final_model.pkl")
joblib.dump(list(X.columns), "pjmw_feature_columns.pkl")
print("Final model saved!")

## Step 15: Prediction on new data (inference)

In [ ]:
def build_features(timestamp, lag_1, lag_24, lag_168, rolling_24, rolling_168):
    ts = pd.Timestamp(timestamp)
    row = {
        "hour": ts.hour,
        "day_of_week": ts.dayofweek,
        "month": ts.month,
        "year": ts.year,
        "is_weekend": int(ts.dayofweek >= 5),
        "hour_sin": np.sin(2 * np.pi * ts.hour / 24),
        "hour_cos": np.cos(2 * np.pi * ts.hour / 24),
        "month_sin": np.sin(2 * np.pi * ts.month / 12),
        "month_cos": np.cos(2 * np.pi * ts.month / 12),
        "lag_1": lag_1, "lag_24": lag_24, "lag_168": lag_168,
        "rolling_24": rolling_24, "rolling_168": rolling_168,
    }
    return pd.DataFrame([row])[joblib.load("pjmw_feature_columns.pkl")]

model = joblib.load("pjmw_final_model.pkl")

x_new = build_features("2018-08-02 14:00",
                       lag_1=6500, lag_24=6400, lag_168=6300,
                       rolling_24=6000, rolling_168=5800)
print("Predicted demand (MW):", model.predict(x_new)[0])